In [1]:
# pip install tonic

In [2]:
import os
import time
from dataclasses import dataclass, asdict, field
from typing import List, Tuple, Optional, Dict, Any, Union, Callable

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from scipy import stats


# ----------------------------- Utilities --------------------------------------
def set_seed(seed: int = 42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def one_hot(y: torch.Tensor, num_classes: int) -> torch.Tensor:
    return F.one_hot(y.long(), num_classes=num_classes).float()


def default_device():
    return "cuda" if torch.cuda.is_available() else "cpu"


# ----------------------------- Metrics ----------------------------------------
class MetricAccumulator:
    def __init__(self, num_classes: int, device: str = "cpu"):
        self.C = int(num_classes)
        self.device = device
        self.reset()

    def reset(self):
        self.tp      = torch.zeros(self.C, dtype=torch.long, device=self.device)
        self.fp      = torch.zeros(self.C, dtype=torch.long, device=self.device)
        self.fn      = torch.zeros(self.C, dtype=torch.long, device=self.device)
        self.correct = 0
        self.total   = 0

    @torch.no_grad()
    def update(self, pred: torch.Tensor, y: torch.Tensor):
        pred = pred.view(-1).long()
        y    = y.view(-1).long()
        self.total   += int(y.numel())
        self.correct += int((pred == y).sum().item())

        pred_count = torch.bincount(pred, minlength=self.C)
        true_count = torch.bincount(y,    minlength=self.C)
        tp         = torch.bincount(pred[pred == y], minlength=self.C)
        self.tp += tp
        self.fp += pred_count - tp
        self.fn += true_count - tp

    @torch.no_grad()
    def compute(self, eps: float = 1e-8) -> Dict[str, float]:
        tp = self.tp.float(); fp = self.fp.float(); fn = self.fn.float()
        p_c = tp / (tp + fp + eps)
        r_c = tp / (tp + fn + eps)
        f_c = 2.0 * p_c * r_c / (p_c + r_c + eps)
        return {
            "acc":       float(self.correct / max(self.total, 1)),
            "precision": float(p_c.mean().item()),
            "recall":    float(r_c.mean().item()),
            "f1":        float(f_c.mean().item()),
        }


def _pretty_metrics(m: Dict[str, float]) -> str:
    return (f"Acc {m['acc']:.4f} | P {m['precision']:.4f} | "
            f"R {m['recall']:.4f} | F1 {m['f1']:.4f}")


# ----------------------------- Early Stopping ---------------------------------
class MetricEarlyStopper:
    def __init__(self, mode: str = "max", patience_epochs: int = 3, min_delta: float = 1e-4):
        assert mode in ("max", "min")
        self.mode       = mode
        self.patience   = int(patience_epochs)
        self.min_delta  = float(min_delta)
        self.best       = None
        self.best_epoch = 0
        self.bad_epochs = 0

    def _is_improvement(self, value: float) -> bool:
        if self.best is None:
            return True
        if self.mode == "max":
            return value > (self.best + self.min_delta)
        return value < (self.best - self.min_delta)

    def step(self, value: float, epoch: int) -> Tuple[bool, bool]:
        improved = self._is_improvement(value)
        if improved:
            self.best       = float(value)
            self.best_epoch = int(epoch)
            self.bad_epochs = 0
        else:
            self.bad_epochs += 1
        return (self.bad_epochs >= self.patience), improved


# ----------------------------- PC Activations ---------------------------------
def make_pc_activation(name: str):
    name = name.lower()
    if name == "sigmoid":
        f      = torch.sigmoid
        fprime = lambda z: torch.sigmoid(z) * (1 - torch.sigmoid(z))
        return f, fprime
    if name == "tanh":
        f      = torch.tanh
        fprime = lambda z: 1 - torch.tanh(z) ** 2
        return f, fprime
    if name == "relu":
        def f(z):      return torch.clamp(z, 0.0, 1.0)
        def fprime(z): return ((z > 0.0) & (z < 1.0)).float()
        return f, fprime
    return (lambda z: z), (lambda z: torch.ones_like(z))


# ============================================================
# HH NEURON
# ============================================================
class HHNeuron(nn.Module):
    class Gate:
        def __init__(self, B, N, device="cpu"):
            self.alpha = torch.zeros(B, N, device=device)
            self.beta  = torch.zeros(B, N, device=device)
            self.state = torch.zeros(B, N, device=device)

        def update(self, dt):
            a = self.alpha * (1.0 - self.state)
            b = self.beta  * self.state
            return torch.clamp(self.state + dt * (a - b), 0.0, 1.0)

        def set_inf(self):
            self.state = self.alpha / (self.alpha + self.beta + 1e-8)

    def __init__(self, N, dt=0.03, device="cpu", thr=10.0, reset=0.0, tau_ref=2.0):
        super().__init__()
        self.N       = N
        self.dt      = float(dt)
        self.device  = device
        self.thr     = float(thr)
        self.reset   = float(reset)
        self.tau_ref = float(tau_ref)

        self.ENa   = nn.Parameter(torch.tensor(115.0), requires_grad=False)
        self.EK    = nn.Parameter(torch.tensor(-12.0), requires_grad=False)
        self.Eleak = nn.Parameter(torch.tensor(10.6),  requires_grad=False)
        self.gNa   = nn.Parameter(torch.tensor(120.0), requires_grad=False)
        self.gK    = nn.Parameter(torch.tensor(36.0),  requires_grad=False)
        self.gLeak = nn.Parameter(torch.tensor(0.3),   requires_grad=False)
        self.Cm    = nn.Parameter(torch.tensor(1.0),   requires_grad=False)

        self.reset_states(B=1)

    def reset_states(self, B: int):
        dev = self.device
        self.B  = B
        self.Vm = torch.zeros(B, self.N, device=dev)
        self.m  = HHNeuron.Gate(B, self.N, dev)
        self.n  = HHNeuron.Gate(B, self.N, dev)
        self.h  = HHNeuron.Gate(B, self.N, dev)
        self._update_gates(self.Vm)
        self.m.set_inf(); self.n.set_inf(); self.h.set_inf()
        self.refr       = torch.zeros(B, self.N, device=dev)
        self.refr_steps = max(1, int(round(self.tau_ref / self.dt)))

    def _update_gates(self, V):
        V = torch.clamp(V, -100.0, 100.0)
        self.n.alpha = 0.01  * (10.0 - V) / (torch.exp((10.0 - V) / 10.0) - 1.0 + 1e-8)
        self.n.beta  = 0.125 * torch.exp(-V / 80.0)
        self.m.alpha = 0.1   * (25.0 - V) / (torch.exp((25.0 - V) / 10.0) - 1.0 + 1e-8)
        self.m.beta  = 4.0   * torch.exp(-V / 18.0)
        self.h.alpha = 0.07  * torch.exp(-V / 20.0)
        self.h.beta  = 1.0   / (torch.exp((30.0 - V) / 10.0) + 1.0)

    def _currents(self, V, I, m, n, h):
        INa = (m**3) * self.gNa * h * (V - self.ENa)
        IK  = (n**4) * self.gK      * (V - self.EK)
        Ile = self.gLeak            * (V - self.Eleak)
        return I - INa - IK - Ile

    def forward(self, I: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        if I.shape[0] != getattr(self, "B", None):
            self.reset_states(B=I.shape[0])
        self._update_gates(self.Vm)
        m = self.m.update(self.dt)
        n = self.n.update(self.dt)
        h = self.h.update(self.dt)

        dV = self._currents(self.Vm, I, m, n, h) / self.Cm
        Vn = self.Vm + self.dt * dV
        Vn = torch.tanh(Vn / 30.0) * 30.0  # soft cap

        can = (self.refr <= 0)
        spk = ((Vn >= self.thr) & can).float()
        self.Vm = torch.where(spk.bool(), torch.full_like(Vn, self.reset), Vn)
        self.m.state = m; self.n.state = n; self.h.state = h

        self.refr = torch.where(
            spk.bool(),
            torch.full_like(self.refr, float(self.refr_steps)),
            torch.clamp(self.refr - 1.0, min=0.0),
        )
        return spk, self.Vm


# ============================================================
# LIF NEURON
# ============================================================
class LIFNeuron(nn.Module):
    def __init__(self, N, dt=0.03, device="cpu", thr=1.0, reset=0.0, tau=0.3, tau_ref=2.0):
        super().__init__()
        self.N       = N
        self.dt      = float(dt)
        self.device  = device
        self.thr     = float(thr)
        self.reset   = float(reset)
        self.tau     = float(tau)
        self.tau_ref = float(tau_ref)
        self.reset_states(B=1)

    def reset_states(self, B: int):
        self.B          = B
        self.Vm         = torch.zeros(B, self.N, device=self.device)
        self.refr       = torch.zeros(B, self.N, device=self.device)
        self.refr_steps = max(1, int(round(self.tau_ref / self.dt)))

    def forward(self, I: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        if I.shape[0] != getattr(self, "B", None):
            self.reset_states(B=I.shape[0])
        can = (self.refr <= 0)
        dV  = (-self.Vm + I) * (self.dt / max(self.tau, 1e-6))
        Vn  = self.Vm + dV
        spk     = ((Vn >= self.thr) & can).float()
        self.Vm = torch.where(spk.bool(), torch.full_like(Vn, self.reset), Vn)
        self.refr = torch.where(
            spk.bool(),
            torch.full_like(self.refr, float(self.refr_steps)),
            torch.clamp(self.refr - 1.0, min=0.0),
        )
        return spk, self.Vm


# ============================================================
# LIF-PC NETWORK
# ============================================================
class PCSNNetLIF(nn.Module):
    def __init__(
        self,
        layer_sizes: List[int],
        dt: float = 0.03,
        device: str = "cuda",
        current_gain: Union[float, List[float]] = 30.0,
        I_bias:       Union[float, List[float]] = 2.0,
        thr_hidden:   Union[float, List[float]] = 0.8,
        lif_tau:      float = 0.3,
        pc_activation: str  = "relu",
        lr:            float = 2e-4,
        weight_decay:  float = 1e-4,
        input_encoding:     str   = "poisson",
        poisson_scale:      float = 1.0,
        precompute_poisson: bool  = True,
    ):
        super().__init__()
        assert len(layer_sizes) >= 2
        self.device  = torch.device(device)
        self.sizes   = layer_sizes
        self.dt      = float(dt)
        self.L       = len(layer_sizes) - 1
        self.S       = self.L
        self.input_encoding     = input_encoding.lower()
        self.poisson_scale      = float(poisson_scale)
        self.precompute_poisson = bool(precompute_poisson)
        assert self.input_encoding in ("poisson", "latency_first", "event_direct")

        self.f, self.fprime = make_pc_activation(pc_activation)

        self.syn = nn.ModuleList([
            nn.Linear(layer_sizes[i], layer_sizes[i + 1], bias=True)
            for i in range(self.L)
        ])
        for lin in self.syn:
            nn.init.xavier_uniform_(lin.weight, gain=0.5)
            nn.init.zeros_(lin.bias)

        def _as_list(v, n):
            if isinstance(v, (list, tuple)):
                v = [float(x) for x in v]
                return v * n if len(v) == 1 else v
            return [float(v)] * n

        self.current_gain = _as_list(current_gain, self.S)
        self.I_bias       = _as_list(I_bias,       self.S)
        self.thr_hidden   = _as_list(thr_hidden,   self.S)

        self.cells = nn.ModuleList([
            LIFNeuron(
                layer_sizes[i + 1], dt=self.dt,
                device=str(self.device), thr=self.thr_hidden[i], tau=lif_tau,
            )
            for i in range(self.S)
        ])
        self.opt = torch.optim.Adam(self.syn.parameters(), lr=lr, weight_decay=weight_decay)
        self._last_spike_sums: Optional[List[torch.Tensor]] = None

    @torch.no_grad()
    def forward_proxies(self, x_in: torch.Tensor, steps_spk: int) -> List[torch.Tensor]:
        proxies, spike_sums = _run_forward_proxies(
            syn=self.syn, neurons=self.cells,
            sizes=self.sizes, S=self.S, f=self.f,
            current_gain=self.current_gain, I_bias=self.I_bias,
            x_in=x_in, steps_spk=steps_spk,
            input_encoding=self.input_encoding,
            poisson_scale=self.poisson_scale,
            precompute_poisson=self.precompute_poisson,
            device=self.device,
        )
        self._last_spike_sums = spike_sums
        return proxies

    @torch.no_grad()
    def last_spike_sums(self) -> Optional[List[torch.Tensor]]:
        return self._last_spike_sums

    def pc_infer(
        self,
        x_init: List[torch.Tensor],
        y_target: Optional[torch.Tensor] = None,
        T_infer: int = 50,
        eta_x:   float = 0.05,
        clamp_output: bool = True,
    ):
        L = self.L
        x = [xi.clone().detach().to(self.device) for xi in x_init]
        x[0] = x[0].clamp(0, 1)
        if clamp_output and (y_target is not None):
            x[L] = y_target.clone().detach().to(self.device).clamp(0, 1)
        z_cache = [None] * L
        for _ in range(T_infer):
            e = [None] * (L + 1)
            e[0] = torch.zeros_like(x[0])
            for l in range(1, L + 1):
                idx       = l - 1
                z         = F.linear(x[l - 1], self.syn[idx].weight, self.syn[idx].bias)
                z_cache[idx] = z
                e[l]      = x[l] - self.f(z)
            for l in range(1, L):
                idx_dn     = l
                downstream = (e[l + 1] * self.fprime(z_cache[idx_dn])) @ self.syn[idx_dn].weight
                x[l]       = (x[l] - eta_x * (e[l] - downstream)).clamp_(0.0, 1.0)
            if not clamp_output:
                x[L] = (x[L] - eta_x * e[L]).clamp_(0.0, 1.0)
        with torch.no_grad():
            energy = 0.0
            for l in range(1, L + 1):
                idx  = l - 1
                z    = F.linear(x[l - 1], self.syn[idx].weight, self.syn[idx].bias)
                el   = x[l] - self.f(z)
                energy += 0.5 * (el ** 2).mean().item()
        return x, e, z_cache, energy

    def pc_learn(self, x, e, z_cache):
        B = x[0].shape[0]
        self.opt.zero_grad()
        for idx in range(self.L):
            local = e[idx + 1] * self.fprime(z_cache[idx])
            self.syn[idx].weight.grad = -(local.T @ x[idx]) / B
            self.syn[idx].bias.grad   = -local.mean(dim=0)
        torch.nn.utils.clip_grad_norm_(self.syn.parameters(), max_norm=1.0)
        self.opt.step()

    def train_step(self, x_in, y_target, steps_spk, T_infer, eta_x):
        proxies = self.forward_proxies(x_in, steps_spk=steps_spk)
        x, e, z_cache, energy = self.pc_infer(
            proxies, y_target=y_target,
            T_infer=T_infer, eta_x=eta_x, clamp_output=True,
        )
        self.pc_learn(x, e, z_cache)
        return energy, proxies


# ============================================================
# SHARED FORWARD-PROXIES CORE
# ============================================================
def _run_forward_proxies(
    syn: nn.ModuleList,
    neurons: nn.ModuleList,
    sizes: List[int],
    S: int,
    f,
    current_gain: List[float],
    I_bias: List[float],
    x_in: torch.Tensor,
    steps_spk: int,
    input_encoding: str,        # "poisson" | "latency_first" | "event_direct"
    poisson_scale: float,
    precompute_poisson: bool,
    device: torch.device,
) -> Tuple[List[torch.Tensor], Optional[List[torch.Tensor]]]:
    if input_encoding == "event_direct":
        assert x_in.dim() == 3, f"event_direct expects (B, T, D), got {x_in.shape}"
        x_event  = x_in.to(device).float().clamp(0, 1)
        T_avail  = x_event.size(1)
        steps_use = min(T_avail, steps_spk)
        x0 = x_event[:, :steps_use, :].mean(dim=1).clamp(0, 1)
        B  = x0.size(0)
    else:
        x0 = x_in.to(device).clamp(0, 1).float()
        B  = x0.size(0)
        steps_use = steps_spk

    for cell in neurons:
        cell.reset_states(B)

    if input_encoding == "latency_first":
        max_time     = float(steps_use)
        lat          = (max_time * (1.0 - x0)).clamp(0.0, max_time)
        tgrid        = torch.arange(1, steps_use + 1, device=device).view(1, 1, -1)
        input_spikes = (lat.unsqueeze(-1) <= tgrid).float()
        input_fired  = torch.zeros_like(x0, dtype=torch.bool, device=device)
        poisson_spikes = None
    elif input_encoding == "poisson":
        input_spikes = None
        input_fired  = None
        if precompute_poisson:
            p = (x0 * poisson_scale).clamp(0.0, 1.0)
            poisson_spikes = (
                torch.rand(B, x0.size(1), steps_use, device=device) < p.unsqueeze(-1)
            ).float()
        else:
            poisson_spikes = None
    else:  # event_direct
        input_spikes   = None
        input_fired    = None
        poisson_spikes = None

    spike_sums = [torch.zeros(B, sizes[i + 1], device=device) for i in range(S)]

    for t in range(steps_use):
        if input_encoding == "event_direct":
            inp = x_event[:, t, :].clamp(0, 1)
        elif input_encoding == "poisson":
            if poisson_spikes is not None:
                inp = poisson_spikes[:, :, t]
            else:
                p   = (x0 * poisson_scale).clamp(0.0, 1.0)
                inp = (torch.rand_like(x0) < p).float()
        else:  # latency_first
            inp = (input_spikes[:, :, t] * (~input_fired)).float()
            input_fired.logical_or_(inp.bool())

        r_prev = inp
        for i in range(S):
            h_val = F.linear(r_prev, syn[i].weight, syn[i].bias)
            I_val = h_val * current_gain[i] + I_bias[i]
            spk, _ = neurons[i](I_val)
            spike_sums[i] += spk
            r_prev = spk

    proxies = [x0]
    for i in range(S):
        proxies.append((spike_sums[i] / float(steps_use)).clamp(0.0, 1.0))
    return proxies, spike_sums


# ============================================================
# HH-PC NETWORK
# ============================================================
class PCSNNetHH(nn.Module):
    def __init__(
        self,
        layer_sizes: List[int],
        dt: float = 0.03,
        device: str = "cuda",
        current_gain: Union[float, List[float]] = 30.0,
        I_bias:        Union[float, List[float]] = 2.0,
        hh_thr:        Union[float, List[float]] = 0.8,
        pc_activation: str = "relu",
        lr:            float = 2e-4,
        weight_decay:  float = 1e-4,
        input_encoding:     str   = "poisson",
        poisson_scale:      float = 1.0,
        precompute_poisson: bool  = True,
    ):
        super().__init__()
        assert len(layer_sizes) >= 2
        self.device  = torch.device(device)
        self.sizes   = layer_sizes
        self.dt      = float(dt)
        self.L       = len(layer_sizes) - 1
        self.S       = self.L
        self.input_encoding     = input_encoding.lower()
        self.poisson_scale      = float(poisson_scale)
        self.precompute_poisson = bool(precompute_poisson)
        assert self.input_encoding in ("poisson", "latency_first", "event_direct")

        self.f, self.fprime = make_pc_activation(pc_activation)

        self.syn = nn.ModuleList([
            nn.Linear(layer_sizes[i], layer_sizes[i + 1], bias=True)
            for i in range(self.L)
        ])
        for lin in self.syn:
            nn.init.xavier_uniform_(lin.weight, gain=0.5)
            nn.init.zeros_(lin.bias)

        def _as_list(v, n):
            if isinstance(v, (list, tuple)):
                v = [float(x) for x in v]
                return v * n if len(v) == 1 else v
            return [float(v)] * n

        self.current_gain = _as_list(current_gain, self.S)
        self.I_bias       = _as_list(I_bias,       self.S)
        self.hh_thr       = _as_list(hh_thr,       self.S)

        self.hh = nn.ModuleList([
            HHNeuron(layer_sizes[i + 1], dt=dt, device=str(self.device), thr=self.hh_thr[i])
            for i in range(self.S)
        ])
        self.opt = torch.optim.Adam(self.syn.parameters(), lr=lr, weight_decay=weight_decay)
        self._last_spike_sums: Optional[List[torch.Tensor]] = None

    @torch.no_grad()
    def forward_proxies(self, x_in: torch.Tensor, steps_spk: int) -> List[torch.Tensor]:
        proxies, spike_sums = _run_forward_proxies(
            syn=self.syn, neurons=self.hh,
            sizes=self.sizes, S=self.S, f=self.f,
            current_gain=self.current_gain, I_bias=self.I_bias,
            x_in=x_in, steps_spk=steps_spk,
            input_encoding=self.input_encoding,
            poisson_scale=self.poisson_scale,
            precompute_poisson=self.precompute_poisson,
            device=self.device,
        )
        self._last_spike_sums = spike_sums
        return proxies

    @torch.no_grad()
    def last_spike_sums(self) -> Optional[List[torch.Tensor]]:
        return self._last_spike_sums

    def pc_infer(
        self,
        x_init: List[torch.Tensor],
        y_target: Optional[torch.Tensor] = None,
        T_infer: int = 50,
        eta_x:   float = 0.05,
        clamp_output: bool = True,
    ):
        L = self.L
        x = [xi.clone().detach().to(self.device) for xi in x_init]
        x[0] = x[0].clamp(0, 1)
        if clamp_output and (y_target is not None):
            x[L] = y_target.clone().detach().to(self.device).clamp(0, 1)

        z_cache = [None] * L
        for _ in range(T_infer):
            e = [None] * (L + 1)
            e[0] = torch.zeros_like(x[0])
            for l in range(1, L + 1):
                idx       = l - 1
                z         = F.linear(x[l - 1], self.syn[idx].weight, self.syn[idx].bias)
                z_cache[idx] = z
                e[l]      = x[l] - self.f(z)
            for l in range(1, L):
                idx_dn     = l
                downstream = (e[l + 1] * self.fprime(z_cache[idx_dn])) @ self.syn[idx_dn].weight
                x[l]       = (x[l] - eta_x * (e[l] - downstream)).clamp_(0.0, 1.0)
            if not clamp_output:
                x[L] = (x[L] - eta_x * e[L]).clamp_(0.0, 1.0)

        with torch.no_grad():
            energy = 0.0
            for l in range(1, L + 1):
                idx  = l - 1
                z    = F.linear(x[l - 1], self.syn[idx].weight, self.syn[idx].bias)
                el   = x[l] - self.f(z)
                energy += 0.5 * (el ** 2).mean().item()
        return x, e, z_cache, energy

    def pc_learn(self, x, e, z_cache):
        B = x[0].shape[0]
        self.opt.zero_grad()
        for idx in range(self.L):
            local = e[idx + 1] * self.fprime(z_cache[idx])
            self.syn[idx].weight.grad = -(local.T @ x[idx]) / B
            self.syn[idx].bias.grad   = -local.mean(dim=0)
        torch.nn.utils.clip_grad_norm_(self.syn.parameters(), max_norm=1.0)
        self.opt.step()

    def train_step(self, x_in, y_target, steps_spk, T_infer, eta_x):
        proxies = self.forward_proxies(x_in, steps_spk=steps_spk)
        x, e, z_cache, energy = self.pc_infer(
            proxies, y_target=y_target,
            T_infer=T_infer, eta_x=eta_x, clamp_output=True,
        )
        self.pc_learn(x, e, z_cache)
        return energy, proxies



# ============================================================
# EVALUATION HELPERS
# ============================================================
@torch.no_grad()
def spike_rate_epoch_any(
    model,
    loader,
    device: str,
    steps_spk: int,
    eval_seed: int = 1234,
    batch_preprocess_fn: Optional[Callable] = None,
) -> Dict[str, Any]:
    model.eval()
    S = getattr(model, "S", 0)
    if S <= 0:
        return {"per_layer": [], "total": 0.0}

    total_spikes = [0.0] * S
    total_denom  = [0.0] * S
    devs = [torch.cuda.current_device()] if str(device).startswith("cuda") and torch.cuda.is_available() else []

    with torch.random.fork_rng(devices=devs, enabled=True):
        torch.manual_seed(eval_seed)
        if devs:
            torch.cuda.manual_seed_all(eval_seed)

        for x, _y in loader:
            if batch_preprocess_fn is not None:
                x = batch_preprocess_fn(x, device)
            else:
                x = x.to(device, non_blocking=True).view(x.size(0), -1)
            B = x.size(0)
            _ = model.forward_proxies(x, steps_spk=steps_spk)
            spike_sums = model.last_spike_sums()
            if spike_sums is None:
                continue
            for li in range(S):
                total_spikes[li] += float(spike_sums[li].sum().item())
                total_denom[li]  += float(B * spike_sums[li].shape[1] * steps_spk)

    per_layer  = [total_spikes[li] / max(total_denom[li], 1.0) for li in range(S)]
    total_rate = sum(total_spikes) / max(sum(total_denom), 1.0)
    return {"per_layer": per_layer, "total": total_rate}


@torch.no_grad()
def eval_epoch_any(
    model,
    loader,
    device: str,
    steps_spk: int,
    T_infer_eval: int,
    eta_x_eval: float,
    eval_mode: str = "pc",
    eval_seed: int = 1234,
    batch_preprocess_fn: Optional[Callable] = None,
) -> Dict[str, float]:
    model.eval()
    C = model.sizes[-1]
    ff_accum   = MetricAccumulator(C, device="cpu")
    mode_accum = MetricAccumulator(C, device="cpu")
    total_energy = 0.0
    total        = 0
    devs = [torch.cuda.current_device()] if str(device).startswith("cuda") and torch.cuda.is_available() else []

    with torch.random.fork_rng(devices=devs, enabled=True):
        torch.manual_seed(eval_seed)
        if devs:
            torch.cuda.manual_seed_all(eval_seed)

        for x, y in loader:
            if batch_preprocess_fn is not None:
                x = batch_preprocess_fn(x, device)
            else:
                x = x.to(device, non_blocking=True).view(x.size(0), -1)
            y = y.to(device, non_blocking=True)
            B = x.size(0)

            proxies = model.forward_proxies(x, steps_spk=steps_spk)
            ff_pred = proxies[-1].argmax(dim=1)
            ff_accum.update(ff_pred.detach().cpu(), y.detach().cpu())

            if eval_mode == "pc":
                x_settle, _, _, _ = model.pc_infer(
                    proxies, y_target=None,
                    T_infer=T_infer_eval, eta_x=eta_x_eval, clamp_output=False,
                )
                mode_pred = x_settle[-1].argmax(dim=1)
            else:
                mode_pred = ff_pred
            mode_accum.update(mode_pred.detach().cpu(), y.detach().cpu())

            y_oh = one_hot(y, C)
            _, _, _, energy = model.pc_infer(
                proxies, y_target=y_oh,
                T_infer=T_infer_eval, eta_x=eta_x_eval, clamp_output=True,
            )
            total_energy += float(energy) * B
            total        += B

    ff_m   = ff_accum.compute()
    mode_m = mode_accum.compute()
    return {
        "ff_acc":        ff_m["acc"],       "ff_precision":   ff_m["precision"],
        "ff_recall":     ff_m["recall"],    "ff_f1":          ff_m["f1"],
        "mode_acc":      mode_m["acc"],     "mode_precision": mode_m["precision"],
        "mode_recall":   mode_m["recall"],  "mode_f1":        mode_m["f1"],
        "pc_energy":     total_energy / max(total, 1),
    }


# ============================================================
# CONFIG + TRAINING LOOP
# ============================================================
@dataclass
class Cfg:
    dataset:     str            = "NMNIST"
    layer_sizes: Tuple[int, ...] = (1156, 512, 10)

    steps_spk:      int   = 25
    input_encoding: str   = "event_direct"
    poisson_scale:  float = 1.0

    current_gain: float = 30.0
    I_bias:       float = 2.0
    thr:          float = 0.8
    lif_tau:      float = 0.3
    pc_activation:  str   = "relu"
    lr:             float = 2e-4
    weight_decay:   float = 1e-4
    T_infer_train:  int   = 100
    T_infer_eval:   int   = 50
    eta_x:          float = 0.05

    epochs:     int = 10
    batch_size: int = 128

    eval_seed: int  = 1234
    eval_mode: str  = "pc"

    patience_epochs: int   = 3
    min_delta:       float = 1e-4

    ckpt_hh:  str = "best_HH_PC.pt"
    ckpt_lif: str = "best_LIF_PC.pt"

    nmnist_merge: str = "sum"

    # Statistical analysis: seeds for multi-run evaluation
    stat_seeds: Tuple[int, ...] = (42, 123, 456, 789, 1011)


def train_model_any(
    model,
    train_loader: DataLoader,
    val_loader:   DataLoader,
    device: str,
    cfg: Cfg,
    ckpt_path: str,
    name: str,
    batch_preprocess_fn: Optional[Callable] = None,
) -> Dict[str, Any]:
    from tqdm import tqdm

    hist: Dict[str, Any] = {
        "train_energy": [], "train_mode": [],
        "val_mode": [],     "val_energy": [],
        "val_spike_rate": [], "epoch_time": [],
    }
    peak_stop = MetricEarlyStopper(
        mode="max",
        patience_epochs=cfg.patience_epochs,
        min_delta=cfg.min_delta,
    )
    os.makedirs(os.path.dirname(ckpt_path) or ".", exist_ok=True)

    print(f"\n=========== TRAINING: {name} ==========")
    print(asdict(cfg))
    print("=" * 42)

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        t0 = time.time()
        epoch_energy_sum = 0.0
        total_seen       = 0
        train_mode_accum = MetricAccumulator(cfg.layer_sizes[-1], device="cpu")

        pbar = tqdm(train_loader, desc=f"{name} Epoch {epoch}/{cfg.epochs}", ncols=120)
        for x_raw, y in pbar:
            if batch_preprocess_fn is not None:
                x = batch_preprocess_fn(x_raw, device)
            else:
                x = x_raw.to(device, non_blocking=True).view(x_raw.size(0), -1)
            y    = y.to(device, non_blocking=True)
            y_oh = one_hot(y, cfg.layer_sizes[-1])

            energy, proxies = model.train_step(
                x, y_oh,
                steps_spk=cfg.steps_spk,
                T_infer=cfg.T_infer_train,
                eta_x=cfg.eta_x,
            )
            B = x.size(0)
            total_seen       += B
            epoch_energy_sum += float(energy) * B

            if cfg.eval_mode == "pc":
                x_settle, _, _, _ = model.pc_infer(
                    proxies, y_target=None,
                    T_infer=max(1, cfg.T_infer_eval // 2),
                    eta_x=cfg.eta_x, clamp_output=False,
                )
                mode_pred = x_settle[-1].argmax(dim=1)
            else:
                mode_pred = proxies[-1].argmax(dim=1)

            train_mode_accum.update(mode_pred.detach().cpu(), y.detach().cpu())
            m_now = train_mode_accum.compute()
            pbar.set_postfix({"E": f"{float(energy):.4f}", "TrAcc": f"{m_now['acc']:.3f}"})

        epoch_time   = time.time() - t0
        train_energy = epoch_energy_sum / max(total_seen, 1)
        train_mode_m = train_mode_accum.compute()

        val_stats = eval_epoch_any(
            model, val_loader, device,
            steps_spk=cfg.steps_spk,
            T_infer_eval=cfg.T_infer_eval,
            eta_x_eval=cfg.eta_x,
            eval_mode=cfg.eval_mode,
            eval_seed=cfg.eval_seed,
            batch_preprocess_fn=batch_preprocess_fn,
        )
        val_spike_rates = spike_rate_epoch_any(
            model, val_loader, device,
            steps_spk=cfg.steps_spk,
            eval_seed=cfg.eval_seed,
            batch_preprocess_fn=batch_preprocess_fn,
        )

        hist["train_energy"].append(train_energy)
        hist["train_mode"].append(train_mode_m)
        hist["val_mode"].append({
            "acc":       val_stats["mode_acc"],
            "precision": val_stats["mode_precision"],
            "recall":    val_stats["mode_recall"],
            "f1":        val_stats["mode_f1"],
        })
        hist["val_energy"].append(val_stats["pc_energy"])
        hist["val_spike_rate"].append(val_spike_rates)
        hist["epoch_time"].append(epoch_time)

        val_mode_m = {
            "acc":       val_stats["mode_acc"],
            "precision": val_stats["mode_precision"],
            "recall":    val_stats["mode_recall"],
            "f1":        val_stats["mode_f1"],
        }
        print(
            f"{name} Ep {epoch:02d} | E {train_energy:.5f} | "
            f"TR [{_pretty_metrics(train_mode_m)}] | "
            f"VAL [{_pretty_metrics(val_mode_m)}] | "
            f"ValE {val_stats['pc_energy']:.5f} | {epoch_time:.1f}s"
        )
        if val_spike_rates["per_layer"]:
            sr_str = "  ".join(
                f"L{li}:{val_spike_rates['per_layer'][li]:.5f}"
                for li in range(len(val_spike_rates["per_layer"]))
            )
            print(f"         | spike-rate/neuron/step: {sr_str} | total: {val_spike_rates['total']:.5f}")

        monitor = float(
            val_stats["mode_acc"] if cfg.eval_mode == "pc" else val_stats["ff_acc"]
        )
        stop, improved = peak_stop.step(monitor, epoch)
        if improved:
            torch.save(model.state_dict(), ckpt_path)
        if stop:
            print(
                f"[EarlyStopping] {name}: best={peak_stop.best:.4f} "
                f"@ epoch {peak_stop.best_epoch}. Stopping."
            )
            break

    if os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        print(f"[restore] {name}: loaded best ckpt from {ckpt_path}")
    else:
        print(f"[warn] No checkpoint found for {name}. Using last weights.")

    return hist


# ============================================================
# DATASET BUILDERS + MODEL FACTORIES
# ============================================================
import torchvision
import torchvision.transforms as T
from torch.utils.data import random_split, Subset


def _split_val(dataset, val_frac=0.1, seed=42):
    """Split a dataset into (train, val) subsets."""
    n     = len(dataset)
    n_val = max(1, int(n * val_frac))
    g     = torch.Generator().manual_seed(seed)
    return random_split(dataset, [n - n_val, n_val], generator=g)


# ------------------------------------------------------------------ MNIST ---
def build_mnist(cfg: Cfg, device: str):
    tf = T.Compose([T.ToTensor(), T.Lambda(lambda x: x.view(-1))])
    full_train = torchvision.datasets.MNIST(
        root="./data", train=True,  download=True, transform=tf)
    test_ds    = torchvision.datasets.MNIST(
        root="./data", train=False, download=True, transform=tf)
    train_ds, val_ds = _split_val(full_train)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader


def build_mnist_model(cfg: Cfg, device: str):
    return PCSNNetHH(
        layer_sizes    = (784, 512, 10),
        dt             = cfg.dt if hasattr(cfg, "dt") else 0.03,
        device         = device,
        current_gain   = cfg.current_gain,
        I_bias         = cfg.I_bias,
        hh_thr         = cfg.thr,
        pc_activation  = cfg.pc_activation,
        lr             = cfg.lr,
        weight_decay   = cfg.weight_decay,
        input_encoding = "poisson",
        poisson_scale  = cfg.poisson_scale,
    )


def build_mnist_lif_model(cfg: Cfg, device: str):
    return PCSNNetLIF(
        layer_sizes    = (784, 512, 10),
        dt             = cfg.dt if hasattr(cfg, "dt") else 0.03,
        device         = device,
        current_gain   = cfg.current_gain,
        I_bias         = cfg.I_bias,
        thr_hidden     = cfg.thr,
        lif_tau        = cfg.lif_tau,
        pc_activation  = cfg.pc_activation,
        lr             = cfg.lr,
        weight_decay   = cfg.weight_decay,
        input_encoding = "poisson",
        poisson_scale  = cfg.poisson_scale,
    )


# -------------------------------------------------------------- FashionMNIST ---
def build_fashionmnist(cfg: Cfg, device: str):
    tf = T.Compose([T.ToTensor(), T.Lambda(lambda x: x.view(-1))])
    full_train = torchvision.datasets.FashionMNIST(
        root="./data", train=True,  download=True, transform=tf)
    test_ds    = torchvision.datasets.FashionMNIST(
        root="./data", train=False, download=True, transform=tf)
    train_ds, val_ds = _split_val(full_train)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader


def build_fashionmnist_model(cfg: Cfg, device: str):
    return PCSNNetHH(
        layer_sizes    = (784, 512, 10),
        dt             = cfg.dt if hasattr(cfg, "dt") else 0.03,
        device         = device,
        current_gain   = cfg.current_gain,
        I_bias         = cfg.I_bias,
        hh_thr         = cfg.thr,
        pc_activation  = cfg.pc_activation,
        lr             = cfg.lr,
        weight_decay   = cfg.weight_decay,
        input_encoding = "poisson",
        poisson_scale  = cfg.poisson_scale,
    )


def build_fashionmnist_lif_model(cfg: Cfg, device: str):
    return PCSNNetLIF(
        layer_sizes    = (784, 512, 10),
        dt             = cfg.dt if hasattr(cfg, "dt") else 0.03,
        device         = device,
        current_gain   = cfg.current_gain,
        I_bias         = cfg.I_bias,
        thr_hidden     = cfg.thr,
        lif_tau        = cfg.lif_tau,
        pc_activation  = cfg.pc_activation,
        lr             = cfg.lr,
        weight_decay   = cfg.weight_decay,
        input_encoding = "poisson",
        poisson_scale  = cfg.poisson_scale,
    )


# ------------------------------------------------------------------ NMNIST ---
def _nmnist_preprocess(x_raw, device):
    """x_raw: (B, T, 2, 34, 34) tonic tensor → (B, T, 1156) merged frames."""
    x = x_raw.to(device).float()
    if x.dim() == 5:                          # (B, T, 2, 34, 34)
        x = x.sum(dim=2)                      # merge polarities → (B, T, 34, 34)
        x = x.view(x.size(0), x.size(1), -1) # → (B, T, 1156)
    return x.clamp(0, 1)


def build_nmnist(cfg: Cfg, device: str):
    try:
        import tonic
        import tonic.transforms as TT
    except ImportError:
        raise ImportError("Install tonic:  pip install tonic")

    n_steps    = cfg.steps_spk
    sensor_size = tonic.datasets.NMNIST.sensor_size          # (34, 34, 2)
    frame_tf   = TT.ToFrame(sensor_size=sensor_size, n_time_bins=n_steps)
    tf         = tonic.transforms.Compose([frame_tf])

    full_train = tonic.datasets.NMNIST(save_to="./data", train=True,  transform=tf)
    test_ds    = tonic.datasets.NMNIST(save_to="./data", train=False, transform=tf)
    train_ds, val_ds = _split_val(full_train)

    def collate(batch):
        xs, ys = zip(*batch)
        return torch.from_numpy(np.stack(xs)), torch.tensor(ys)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  collate_fn=collate, num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False, collate_fn=collate, num_workers=2)
    test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False, collate_fn=collate, num_workers=2)
    return train_loader, val_loader, test_loader


def build_nmnist_model(cfg: Cfg, device: str):
    return PCSNNetHH(
        layer_sizes    = (1156, 512, 10),
        dt             = cfg.dt if hasattr(cfg, "dt") else 0.03,
        device         = device,
        current_gain   = cfg.current_gain,
        I_bias         = cfg.I_bias,
        hh_thr         = cfg.thr,
        pc_activation  = cfg.pc_activation,
        lr             = cfg.lr,
        weight_decay   = cfg.weight_decay,
        input_encoding = "event_direct",
        poisson_scale  = cfg.poisson_scale,
    )


def build_nmnist_lif_model(cfg: Cfg, device: str):
    return PCSNNetLIF(
        layer_sizes    = (1156, 512, 10),
        dt             = cfg.dt if hasattr(cfg, "dt") else 0.03,
        device         = device,
        current_gain   = cfg.current_gain,
        I_bias         = cfg.I_bias,
        thr_hidden     = cfg.thr,
        lif_tau        = cfg.lif_tau,
        pc_activation  = cfg.pc_activation,
        lr             = cfg.lr,
        weight_decay   = cfg.weight_decay,
        input_encoding = "event_direct",
        poisson_scale  = cfg.poisson_scale,
    )


# ------------------------------------------------------------------ CALTECH ---
def build_caltech(cfg: Cfg, device: str):
    """
    Binary classification: Faces_easy (label 0) vs Motorbikes (label 1).
    Images resized to 32x32 grayscale → 1024-dim flat vector.
    """
    tf = T.Compose([
        T.Resize((32, 32)),
        T.Grayscale(num_output_channels=1),
        T.ToTensor(),
        T.Lambda(lambda x: x.view(-1)),
    ])
    full_ds = torchvision.datasets.Caltech101(
        root="./data", download=True, transform=tf)

    # Filter to only Faces_easy and Motorbikes
    cat_to_idx = {c: i for i, c in enumerate(full_ds.categories)}
    face_idx   = cat_to_idx["Faces_easy"]
    moto_idx   = cat_to_idx["Motorbikes"]

    indices = [i for i, lbl in enumerate(full_ds.y)
               if lbl in (face_idx, moto_idx)]
    label_map = {face_idx: 0, moto_idx: 1}

    class RemappedSubset(torch.utils.data.Dataset):
        def __init__(self, ds, idxs, lmap):
            self.ds = ds; self.idxs = idxs; self.lmap = lmap
        def __len__(self): return len(self.idxs)
        def __getitem__(self, i):
            x, y = self.ds[self.idxs[i]]
            return x, self.lmap[y]

    binary_ds = RemappedSubset(full_ds, indices, label_map)
    n         = len(binary_ds)
    n_test    = max(1, int(n * 0.15))
    n_val     = max(1, int(n * 0.10))
    n_train   = n - n_val - n_test
    g         = torch.Generator().manual_seed(42)
    train_ds, val_ds, test_ds = random_split(
        binary_ds, [n_train, n_val, n_test], generator=g)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader


def build_caltech_model(cfg: Cfg, device: str):
    return PCSNNetHH(
        layer_sizes    = (1024, 512, 2),
        dt             = cfg.dt if hasattr(cfg, "dt") else 0.03,
        device         = device,
        current_gain   = cfg.current_gain,
        I_bias         = cfg.I_bias,
        hh_thr         = cfg.thr,
        pc_activation  = cfg.pc_activation,
        lr             = cfg.lr,
        weight_decay   = cfg.weight_decay,
        input_encoding = "poisson",
        poisson_scale  = cfg.poisson_scale,
    )


def build_caltech_lif_model(cfg: Cfg, device: str):
    return PCSNNetLIF(
        layer_sizes    = (1024, 512, 2),
        dt             = cfg.dt if hasattr(cfg, "dt") else 0.03,
        device         = device,
        current_gain   = cfg.current_gain,
        I_bias         = cfg.I_bias,
        thr_hidden     = cfg.thr,
        lif_tau        = cfg.lif_tau,
        pc_activation  = cfg.pc_activation,
        lr             = cfg.lr,
        weight_decay   = cfg.weight_decay,
        input_encoding = "poisson",
        poisson_scale  = cfg.poisson_scale,
    )


# ------------------------------------------------------------------ REGISTRY ---
DATASET_REGISTRY = {
    "MNIST":        (build_mnist_model,        build_mnist_lif_model,        build_mnist,        None),
    "FashionMNIST": (build_fashionmnist_model,  build_fashionmnist_lif_model, build_fashionmnist, None),
    "NMNIST":       (build_nmnist_model,        build_nmnist_lif_model,       build_nmnist,       _nmnist_preprocess),
    "Caltech":      (build_caltech_model,       build_caltech_lif_model,      build_caltech,      None),
}


def run_stat_analysis_for_dataset(dataset_name: str, cfg: Cfg, device: str):
    """
    Convenience wrapper. Example:
        cfg = Cfg(dataset="MNIST", layer_sizes=(784, 512, 10))
        run_stat_analysis_for_dataset("MNIST", cfg, device="cuda")
    """
    assert dataset_name in DATASET_REGISTRY, \
        f"Unknown dataset '{dataset_name}'. Choose from {list(DATASET_REGISTRY)}"
    hh_fn, lif_fn, loader_fn, pre_fn = DATASET_REGISTRY[dataset_name]
    return run_statistical_analysis(
        build_hh_model_fn  = lambda: hh_fn(cfg, device),
        build_lif_model_fn = lambda: lif_fn(cfg, device),
        build_loaders_fn   = lambda: loader_fn(cfg, device),
        cfg                = cfg,
        device             = device,
        batch_preprocess_fn = pre_fn,
    )


# ============================================================
# STATISTICAL ANALYSIS  (Section 5.3)
# ============================================================
def run_statistical_analysis(
    build_hh_model_fn,
    build_lif_model_fn,
    build_loaders_fn,
    cfg: Cfg,
    device: str,
    batch_preprocess_fn=None,
) -> Dict[str, Any]:
    """
    Train HH+PC and LIF+PC independently for each seed in cfg.stat_seeds.
    For every seed, record final test accuracy and F1 for both models.
    Then run a paired t-test (HH+PC vs LIF+PC) across seeds.

    Returns dict with per_seed results, summary stats, and t-test results.
    """
    per_seed_results: List[Dict[str, float]] = []

    for seed in cfg.stat_seeds:
        print(f"\n{'='*60}")
        print(f"  Statistical run  |  seed = {seed}")
        print(f"{'='*60}")

        # ---------- HH + PC ----------
        set_seed(seed)
        hh_model = build_hh_model_fn().to(device)
        train_loader, val_loader, test_loader = build_loaders_fn()
        ckpt_hh = cfg.ckpt_hh.replace(".pt", f"_seed{seed}.pt")
        train_model_any(
            hh_model, train_loader, val_loader,
            device=device, cfg=cfg, ckpt_path=ckpt_hh,
            name=f"HH-PC[seed={seed}]",
            batch_preprocess_fn=batch_preprocess_fn,
        )
        hh_stats = eval_epoch_any(
            hh_model, test_loader, device,
            steps_spk=cfg.steps_spk,
            T_infer_eval=cfg.T_infer_eval,
            eta_x_eval=cfg.eta_x,
            eval_mode="pc",
            eval_seed=seed,
            batch_preprocess_fn=batch_preprocess_fn,
        )

        # ---------- LIF + PC ----------
        set_seed(seed)
        lif_model = build_lif_model_fn().to(device)
        train_loader, val_loader, test_loader = build_loaders_fn()
        ckpt_lif = cfg.ckpt_lif.replace(".pt", f"_seed{seed}.pt")
        train_model_any(
            lif_model, train_loader, val_loader,
            device=device, cfg=cfg, ckpt_path=ckpt_lif,
            name=f"LIF-PC[seed={seed}]",
            batch_preprocess_fn=batch_preprocess_fn,
        )
        lif_stats = eval_epoch_any(
            lif_model, test_loader, device,
            steps_spk=cfg.steps_spk,
            T_infer_eval=cfg.T_infer_eval,
            eta_x_eval=cfg.eta_x,
            eval_mode="pc",
            eval_seed=seed,
            batch_preprocess_fn=batch_preprocess_fn,
        )

        per_seed_results.append({
            "seed":      seed,
            "hh_acc":    hh_stats["mode_acc"],
            "hh_f1":     hh_stats["mode_f1"],
            "hh_energy": hh_stats["pc_energy"],
            "lif_acc":   lif_stats["mode_acc"],
            "lif_f1":    lif_stats["mode_f1"],
            "lif_energy":lif_stats["pc_energy"],
        })

    # ---------- aggregate ----------
    def _col(key):
        return np.array([r[key] for r in per_seed_results], dtype=np.float64)

    hh_accs  = _col("hh_acc");   lif_accs  = _col("lif_acc")
    hh_f1s   = _col("hh_f1");    lif_f1s   = _col("lif_f1")
    hh_enrgs = _col("hh_energy"); lif_enrgs = _col("lif_energy")

    summary = {
        "hh_acc_mean":     float(hh_accs.mean()),   "hh_acc_std":     float(hh_accs.std()),
        "hh_f1_mean":      float(hh_f1s.mean()),    "hh_f1_std":      float(hh_f1s.std()),
        "hh_energy_mean":  float(hh_enrgs.mean()),  "hh_energy_std":  float(hh_enrgs.std()),
        "lif_acc_mean":    float(lif_accs.mean()),  "lif_acc_std":    float(lif_accs.std()),
        "lif_f1_mean":     float(lif_f1s.mean()),   "lif_f1_std":     float(lif_f1s.std()),
        "lif_energy_mean": float(lif_enrgs.mean()), "lif_energy_std": float(lif_enrgs.std()),
    }

    # ---------- paired t-tests: HH+PC vs LIF+PC ----------
    t_acc, p_acc = stats.ttest_rel(hh_accs, lif_accs)
    t_f1,  p_f1  = stats.ttest_rel(hh_f1s,  lif_f1s)

    # ---------- print summary ----------
    n = len(cfg.stat_seeds)
    print(f"\n{'='*70}")
    print(f"  STATISTICAL ANALYSIS  |  HH+PC vs LIF+PC  |  {n} seeds")
    print(f"{'='*70}")
    print(f"{'Seed':<8} {'HH Acc':>8} {'LIF Acc':>8} {'HH F1':>8} {'LIF F1':>8} {'HH Nrg':>9} {'LIF Nrg':>9}")
    print(f"{'-'*65}")
    for r in per_seed_results:
        print(
            f"{r['seed']:<8} {r['hh_acc']:>8.4f} {r['lif_acc']:>8.4f} "
            f"{r['hh_f1']:>8.4f} {r['lif_f1']:>8.4f} "
            f"{r['hh_energy']:>9.5f} {r['lif_energy']:>9.5f}"
        )
    print(f"{'-'*65}")
    print(
        f"{'Mean':<8} {summary['hh_acc_mean']:>8.4f} {summary['lif_acc_mean']:>8.4f} "
        f"{summary['hh_f1_mean']:>8.4f} {summary['lif_f1_mean']:>8.4f} "
        f"{summary['hh_energy_mean']:>9.5f} {summary['lif_energy_mean']:>9.5f}"
    )
    print(
        f"{'Std':<8} {summary['hh_acc_std']:>8.4f} {summary['lif_acc_std']:>8.4f} "
        f"{summary['hh_f1_std']:>8.4f} {summary['lif_f1_std']:>8.4f} "
        f"{summary['hh_energy_std']:>9.5f} {summary['lif_energy_std']:>9.5f}"
    )
    print(f"\n  Paired t-test (HH+PC vs LIF+PC) across {n} seeds:")
    print(f"    Accuracy : t = {t_acc:+.4f},  p = {p_acc:.4f}  "
          f"{'*** significant (p<0.05)' if p_acc < 0.05 else '(not significant)'}")
    print(f"    F1 Score : t = {t_f1:+.4f},  p = {p_f1:.4f}  "
          f"{'*** significant (p<0.05)' if p_f1  < 0.05 else '(not significant)'}")
    print(f"{'='*70}\n")

    return {
        "per_seed":  per_seed_results,
        "summary":   summary,
        "ttest_acc": (float(t_acc), float(p_acc)),
        "ttest_f1":  (float(t_f1),  float(p_f1)),
    }

In [3]:
device = default_device()

In [4]:
# run_stat_analysis_for_dataset("NMNIST",       Cfg(dataset="NMNIST", ckpt_hh="best_NMNIST.pt"),       device)

In [5]:
cfg    = Cfg(dataset="MNIST", ckpt_hh="best_MNIST.pt", ckpt_lif="best_MNIST_LIF_PC.pt")
run_stat_analysis_for_dataset("MNIST", cfg, device)



  Statistical run  |  seed = 42


100%|██████████| 9.91M/9.91M [00:00<00:00, 41.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.04MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.3MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.24MB/s]



=========== TRAINING: HH-PC[seed=42] ==========
{'dataset': 'MNIST', 'layer_sizes': (1156, 512, 10), 'steps_spk': 25, 'input_encoding': 'event_direct', 'poisson_scale': 1.0, 'current_gain': 30.0, 'I_bias': 2.0, 'thr': 0.8, 'lif_tau': 0.3, 'pc_activation': 'relu', 'lr': 0.0002, 'weight_decay': 0.0001, 'T_infer_train': 100, 'T_infer_eval': 50, 'eta_x': 0.05, 'epochs': 10, 'batch_size': 128, 'eval_seed': 1234, 'eval_mode': 'pc', 'patience_epochs': 3, 'min_delta': 0.0001, 'ckpt_hh': 'best_MNIST.pt', 'ckpt_lif': 'best_MNIST_LIF_PC.pt', 'nmnist_merge': 'sum', 'stat_seeds': (42, 123, 456, 789, 1011)}


HH-PC[seed=42] Epoch 1/10: 100%|███████████████████████████████| 422/422 [00:52<00:00,  8.11it/s, E=0.0020, TrAcc=0.901]


HH-PC[seed=42] Ep 01 | E 0.00457 | TR [Acc 0.9006 | P 0.9002 | R 0.8991 | F1 0.8992] | VAL [Acc 0.9417 | P 0.9413 | R 0.9403 | F1 0.9407] | ValE 0.00181 | 52.1s
         | spike-rate/neuron/step: L0:0.01682  L1:0.01864 | total: 0.01686


...

LIF-PC[seed=1011] Epoch 9/10: 100%|████████████████████████████| 422/422 [00:33<00:00, 12.62it/s, E=0.0001, TrAcc=0.983]


LIF-PC[seed=1011] Ep 09 | E 0.00013 | TR [Acc 0.9828 | P 0.9827 | R 0.9827 | F1 0.9827] | VAL [Acc 0.9742 | P 0.9740 | R 0.9738 | F1 0.9738] | ValE 0.00015 | 33.4s
         | spike-rate/neuron/step: L0:0.01918  L1:0.03274 | total: 0.01944


LIF-PC[seed=1011] Epoch 10/10: 100%|███████████████████████████| 422/422 [00:33<00:00, 12.74it/s, E=0.0005, TrAcc=0.983]


LIF-PC[seed=1011] Ep 10 | E 0.00013 | TR [Acc 0.9828 | P 0.9828 | R 0.9827 | F1 0.9828] | VAL [Acc 0.9710 | P 0.9708 | R 0.9706 | F1 0.9706] | ValE 0.00018 | 33.1s
         | spike-rate/neuron/step: L0:0.01962  L1:0.03211 | total: 0.01986
[restore] LIF-PC[seed=1011]: loaded best ckpt from best_MNIST_LIF_PC_seed1011.pt

  STATISTICAL ANALYSIS  |  HH+PC vs LIF+PC  |  5 seeds
Seed       HH Acc  LIF Acc    HH F1   LIF F1    HH Nrg   LIF Nrg
-----------------------------------------------------------------
42         0.9753   0.9755   0.9752   0.9754   0.00016   0.00017
123        0.9748   0.9741   0.9747   0.9740   0.00014   0.00022
456        0.9738   0.9747   0.9737   0.9746   0.00015   0.00016
789        0.9755   0.9731   0.9754   0.9730   0.00015   0.00028
1011       0.9749   0.9744   0.9748   0.9743   0.00017   0.00017
-----------------------------------------------------------------
Mean       0.9749   0.9744   0.9747   0.9742   0.00016   0.00020
Std        0.0006   0.0008   0.0006  

{'per_seed': [{'seed': 42,
   'hh_acc': 0.9753,
   'hh_f1': 0.9751943349838257,
   'hh_energy': 0.00016145103590606595,
   'lif_acc': 0.9755,
   'lif_f1': 0.9753939509391785,
   'lif_energy': 0.00016838300736199017},
  {'seed': 123,
   'hh_acc': 0.9748,
   'hh_f1': 0.9746729731559753,
   'hh_energy': 0.00014279115342069417,
   'lif_acc': 0.9741,
   'lif_f1': 0.9739724397659302,
   'lif_energy': 0.0002209455050775432},
  {'seed': 456,
   'hh_acc': 0.9738,
   'hh_f1': 0.9736565351486206,
   'hh_energy': 0.0001536145506601315,
   'lif_acc': 0.9747,
   'lif_f1': 0.9746376276016235,
   'lif_energy': 0.00015677174594675306},
  {'seed': 789,
   'hh_acc': 0.9755,
   'hh_f1': 0.9753792881965637,
   'hh_energy': 0.00014543923485907726,
   'lif_acc': 0.9731,
   'lif_f1': 0.9729577302932739,
   'lif_energy': 0.00028428360892867205},
  {'seed': 1011,
   'hh_acc': 0.9749,
   'hh_f1': 0.9747945666313171,
   'hh_energy': 0.00017212855511752423,
   'lif_acc': 0.9744,
   'lif_f1': 0.9742825627326965,
  

In [6]:
run_stat_analysis_for_dataset("FashionMNIST", Cfg(dataset="FashionMNIST", ckpt_hh="best_FashionMNIST.pt", ckpt_lif="best_FMNIST_LIF_PC.pt"), device)


  Statistical run  |  seed = 42


100%|██████████| 26.4M/26.4M [00:02<00:00, 12.6MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 273kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.05MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 11.4MB/s]



=========== TRAINING: HH-PC[seed=42] ==========
{'dataset': 'FashionMNIST', 'layer_sizes': (1156, 512, 10), 'steps_spk': 25, 'input_encoding': 'event_direct', 'poisson_scale': 1.0, 'current_gain': 30.0, 'I_bias': 2.0, 'thr': 0.8, 'lif_tau': 0.3, 'pc_activation': 'relu', 'lr': 0.0002, 'weight_decay': 0.0001, 'T_infer_train': 100, 'T_infer_eval': 50, 'eta_x': 0.05, 'epochs': 10, 'batch_size': 128, 'eval_seed': 1234, 'eval_mode': 'pc', 'patience_epochs': 3, 'min_delta': 0.0001, 'ckpt_hh': 'best_FashionMNIST.pt', 'ckpt_lif': 'best_FMNIST_LIF_PC.pt', 'nmnist_merge': 'sum', 'stat_seeds': (42, 123, 456, 789, 1011)}


HH-PC[seed=42] Epoch 1/10: 100%|███████████████████████████████| 422/422 [00:51<00:00,  8.19it/s, E=0.0023, TrAcc=0.803]


HH-PC[seed=42] Ep 01 | E 0.00653 | TR [Acc 0.8032 | P 0.8041 | R 0.8030 | F1 0.7983] | VAL [Acc 0.8510 | P 0.8515 | R 0.8521 | F1 0.8488] | ValE 0.00255 | 51.6s
         | spike-rate/neuron/step: L0:0.01784  L1:0.02542 | total: 0.01798


...

LIF-PC[seed=1011] Epoch 9/10: 100%|████████████████████████████| 422/422 [00:34<00:00, 12.39it/s, E=0.0000, TrAcc=0.900]


LIF-PC[seed=1011] Ep 09 | E 0.00023 | TR [Acc 0.9002 | P 0.8988 | R 0.9001 | F1 0.8987] | VAL [Acc 0.8768 | P 0.8794 | R 0.8781 | F1 0.8728] | ValE 0.00027 | 34.1s
         | spike-rate/neuron/step: L0:0.02034  L1:0.03640 | total: 0.02065


LIF-PC[seed=1011] Epoch 10/10: 100%|███████████████████████████| 422/422 [00:34<00:00, 12.37it/s, E=0.0001, TrAcc=0.902]


LIF-PC[seed=1011] Ep 10 | E 0.00021 | TR [Acc 0.9023 | P 0.9011 | R 0.9022 | F1 0.9007] | VAL [Acc 0.8807 | P 0.8810 | R 0.8818 | F1 0.8801] | ValE 0.00023 | 34.1s
         | spike-rate/neuron/step: L0:0.01993  L1:0.03629 | total: 0.02024
[restore] LIF-PC[seed=1011]: loaded best ckpt from best_FMNIST_LIF_PC_seed1011.pt

  STATISTICAL ANALYSIS  |  HH+PC vs LIF+PC  |  5 seeds
Seed       HH Acc  LIF Acc    HH F1   LIF F1    HH Nrg   LIF Nrg
-----------------------------------------------------------------
42         0.8767   0.8724   0.8758   0.8703   0.00026   0.00045
123        0.8765   0.8749   0.8746   0.8738   0.00034   0.00029
456        0.8757   0.8697   0.8732   0.8672   0.00032   0.00048
789        0.8685   0.8663   0.8656   0.8630   0.00063   0.00052
1011       0.8777   0.8765   0.8742   0.8753   0.00031   0.00025
-----------------------------------------------------------------
Mean       0.8750   0.8720   0.8727   0.8699   0.00037   0.00040
Std        0.0033   0.0036   0.0037 

{'per_seed': [{'seed': 42,
   'hh_acc': 0.8767,
   'hh_f1': 0.8757994771003723,
   'hh_energy': 0.00026216405443265105,
   'lif_acc': 0.8724,
   'lif_f1': 0.8702691793441772,
   'lif_energy': 0.000453490161919035},
  {'seed': 123,
   'hh_acc': 0.8765,
   'hh_f1': 0.8745796084403992,
   'hh_energy': 0.00034247335350082723,
   'lif_acc': 0.8749,
   'lif_f1': 0.8737813830375671,
   'lif_energy': 0.0002934116959309904},
  {'seed': 456,
   'hh_acc': 0.8757,
   'hh_f1': 0.87324059009552,
   'hh_energy': 0.00032121967949788084,
   'lif_acc': 0.8697,
   'lif_f1': 0.8671571612358093,
   'lif_energy': 0.0004760663303954061},
  {'seed': 789,
   'hh_acc': 0.8685,
   'hh_f1': 0.8655511140823364,
   'hh_energy': 0.0006327870838460512,
   'lif_acc': 0.8663,
   'lif_f1': 0.8630242347717285,
   'lif_energy': 0.0005166254301846493},
  {'seed': 1011,
   'hh_acc': 0.8777,
   'hh_f1': 0.8741706609725952,
   'hh_energy': 0.00030581710715487135,
   'lif_acc': 0.8765,
   'lif_f1': 0.8753353357315063,
   'lif_

In [7]:
run_stat_analysis_for_dataset("Caltech", Cfg(dataset="Caltech", layer_sizes=(1024, 512, 2), ckpt_hh="best_Caltech.pt", ckpt_lif="best_Caltech_LIF_PC.pt"), device)


  Statistical run  |  seed = 42


100%|██████████| 137M/137M [00:03<00:00, 41.8MB/s]



=========== TRAINING: HH-PC[seed=42] ==========
{'dataset': 'Caltech', 'layer_sizes': (1024, 512, 2), 'steps_spk': 25, 'input_encoding': 'event_direct', 'poisson_scale': 1.0, 'current_gain': 30.0, 'I_bias': 2.0, 'thr': 0.8, 'lif_tau': 0.3, 'pc_activation': 'relu', 'lr': 0.0002, 'weight_decay': 0.0001, 'T_infer_train': 100, 'T_infer_eval': 50, 'eta_x': 0.05, 'epochs': 10, 'batch_size': 128, 'eval_seed': 1234, 'eval_mode': 'pc', 'patience_epochs': 3, 'min_delta': 0.0001, 'ckpt_hh': 'best_Caltech.pt', 'ckpt_lif': 'best_Caltech_LIF_PC.pt', 'nmnist_merge': 'sum', 'stat_seeds': (42, 123, 456, 789, 1011)}


HH-PC[seed=42] Epoch 1/10: 100%|███████████████████████████████████| 8/8 [00:01<00:00,  5.35it/s, E=0.0252, TrAcc=0.676]


HH-PC[seed=42] Ep 01 | E 0.12769 | TR [Acc 0.6760 | P 0.7166 | R 0.7253 | F1 0.6756] | VAL [Acc 0.9593 | P 0.9747 | R 0.9138 | F1 0.9399] | ValE 0.01578 | 1.5s
         | spike-rate/neuron/step: L0:0.01286  L1:0.04000 | total: 0.01297


...

LIF-PC[seed=1011] Epoch 5/10: 100%|████████████████████████████████| 8/8 [00:00<00:00,  8.63it/s, E=0.0027, TrAcc=0.976]


LIF-PC[seed=1011] Ep 05 | E 0.00536 | TR [Acc 0.9762 | P 0.9729 | R 0.9755 | F1 0.9742] | VAL [Acc 1.0000 | P 1.0000 | R 1.0000 | F1 1.0000] | ValE 0.00230 | 0.9s
         | spike-rate/neuron/step: L0:0.01441  L1:0.04000 | total: 0.01451


LIF-PC[seed=1011] Epoch 6/10: 100%|████████████████████████████████| 8/8 [00:00<00:00,  8.31it/s, E=0.0029, TrAcc=0.982]


LIF-PC[seed=1011] Ep 06 | E 0.00449 | TR [Acc 0.9816 | P 0.9785 | R 0.9817 | F1 0.9801] | VAL [Acc 1.0000 | P 1.0000 | R 1.0000 | F1 1.0000] | ValE 0.00208 | 1.0s
         | spike-rate/neuron/step: L0:0.01432  L1:0.04000 | total: 0.01442
[EarlyStopping] LIF-PC[seed=1011]: best=1.0000 @ epoch 3. Stopping.
[restore] LIF-PC[seed=1011]: loaded best ckpt from best_Caltech_LIF_PC_seed1011.pt

  STATISTICAL ANALYSIS  |  HH+PC vs LIF+PC  |  5 seeds
Seed       HH Acc  LIF Acc    HH F1   LIF F1    HH Nrg   LIF Nrg
-----------------------------------------------------------------
42         0.9728   0.9783   0.9720   0.9776   0.00788   0.00768
123        0.9783   0.9783   0.9775   0.9775   0.00569   0.00585
456        0.9783   0.9783   0.9775   0.9775   0.00615   0.00621
789        0.9837   0.9837   0.9832   0.9832   0.00546   0.00543
1011       0.9783   0.9783   0.9776   0.9776   0.00685   0.00687
-----------------------------------------------------------------
Mean       0.9783   0.9793   0.97

{'per_seed': [{'seed': 42,
   'hh_acc': 0.9728260869565217,
   'hh_f1': 0.9720322489738464,
   'hh_energy': 0.007884441587046476,
   'lif_acc': 0.9782608695652174,
   'lif_f1': 0.9775828123092651,
   'lif_energy': 0.0076814014648097445},
  {'seed': 123,
   'hh_acc': 0.9782608695652174,
   'hh_f1': 0.9774923324584961,
   'hh_energy': 0.0056851095462701305,
   'lif_acc': 0.9782608695652174,
   'lif_f1': 0.9774923324584961,
   'lif_energy': 0.005850410488599901},
  {'seed': 456,
   'hh_acc': 0.9782608695652174,
   'hh_f1': 0.9774923324584961,
   'hh_energy': 0.006152805449465118,
   'lif_acc': 0.9782608695652174,
   'lif_f1': 0.9774923324584961,
   'lif_energy': 0.0062134953566229615},
  {'seed': 789,
   'hh_acc': 0.9836956521739131,
   'hh_f1': 0.983153760433197,
   'hh_energy': 0.005459724512073318,
   'lif_acc': 0.9836956521739131,
   'lif_f1': 0.983153760433197,
   'lif_energy': 0.0054332301139726515},
  {'seed': 1011,
   'hh_acc': 0.9782608695652174,
   'hh_f1': 0.9775828123092651,
 